## Installation instructions

Keep `romanian_constitution.txt` in the same directory as this notebook.

In [ ]:
%pip install torch transformers

# TASK1

In [1]:
original_corpus = [
    "low",
    "new",
    "lowest",
    "newest",
    "widest",
    "lower",
]

In [2]:
def get_vocabulary(corpus: list[str]) -> list[str]:
    return {letter for word in corpus for letter in word} 

In [3]:
def process_corpus(corpus: list[str]) -> list[list[str]]:
    return [[letter for letter in word] for word in corpus]

In [4]:
def count_vocab(corpus: list[list[str]]) -> dict[str, list[int, str, str]]:
    result = {}
    for word in corpus:
        for i in range(len(word) - 1):
            pair = word[i] + word[i+1]
            if pair in result: 
                result[pair][0] += 1
            else:
                result[pair] = [1, word[i], word[i+1]]

    return result

In [5]:
def get_most_freq_pair(counts: dict[str, list[int, str, str]]) -> tuple[str, str, str]:
    max_freq = 0
    max_pair = None
    for key, value in counts.items(): 
        if value[0] > max_freq:
            max_freq = value[0]
            max_pair = (key, value[1], value[2])

    return max_pair

In [6]:
def merge_pair(corpus: list[list[str]], pair: str) -> list[list[str]]:
    result = []
    for word in corpus:
        new_word = []
        i = 0
        while i < len(word) - 1:
            
            if word[i] + word[i+1] == pair:
                new_word.append(pair)
                i+=2
            else:
                new_word.append(word[i])
                i+=1

        if i < len(word):
            new_word.append(word[i])

        result.append(new_word)

    return result

In [7]:
def bpe(corpus: list[str]) -> list[tuple[str, str]]:
    vocab = get_vocabulary(corpus)
    corpus = process_corpus(corpus)
    merges = []

    for i in range(6):
        counts = count_vocab(corpus)
        pair, a, b = get_most_freq_pair(counts)
        vocab.add(pair)
        corpus = merge_pair(corpus, pair)
        merges.append((a, b))

        print(f"Merge {i+1}")
        print(f"Seleceted pair: {a} + {b} = {pair}")
        print(f"New vocab:\n{vocab}")
        print(f"New corpus:\n{corpus}")

    return merges
    

In [8]:
merges = bpe(original_corpus)

Merge 1
Seleceted pair: l + o = lo
New vocab:
{'s', 'i', 'd', 'n', 'lo', 'o', 'w', 'e', 'r', 't', 'l'}
New corpus:
[['lo', 'w'], ['n', 'e', 'w'], ['lo', 'w', 'e', 's', 't'], ['n', 'e', 'w', 'e', 's', 't'], ['w', 'i', 'd', 'e', 's', 't'], ['lo', 'w', 'e', 'r']]
Merge 2
Seleceted pair: lo + w = low
New vocab:
{'s', 'i', 'd', 'n', 'lo', 'o', 'w', 'low', 'e', 'r', 't', 'l'}
New corpus:
[['low'], ['n', 'e', 'w'], ['low', 'e', 's', 't'], ['n', 'e', 'w', 'e', 's', 't'], ['w', 'i', 'd', 'e', 's', 't'], ['low', 'e', 'r']]
Merge 3
Seleceted pair: e + s = es
New vocab:
{'s', 'i', 'es', 'd', 'n', 'lo', 'o', 'w', 'low', 'e', 'r', 't', 'l'}
New corpus:
[['low'], ['n', 'e', 'w'], ['low', 'es', 't'], ['n', 'e', 'w', 'es', 't'], ['w', 'i', 'd', 'es', 't'], ['low', 'e', 'r']]
Merge 4
Seleceted pair: es + t = est
New vocab:
{'s', 'i', 'es', 'd', 'n', 'est', 'lo', 'o', 'w', 'low', 'e', 'r', 't', 'l'}
New corpus:
[['low'], ['n', 'e', 'w'], ['low', 'est'], ['n', 'e', 'w', 'est'], ['w', 'i', 'd', 'est'], ['l

In [9]:
def test_word(word: str, merges: list[tuple[str, str]]):
    word = [letter for letter in word]
    found = True
    
    while found:
        print(word)
        found = False
        new_word = []
        i = 0
        while i < len(word) - 1:
            if not found and (word[i], word[i+1]) in merges:
                found = True
                new_word.append(word[i] + word[i+1])
                i+=2
            else:
                new_word.append(word[i])
                i+=1

        if i < len(word):
            new_word.append(word[i])

        word = new_word

In [10]:
test_word("slowest", merges)

['s', 'l', 'o', 'w', 'e', 's', 't']
['s', 'lo', 'w', 'e', 's', 't']
['s', 'low', 'e', 's', 't']
['s', 'low', 'es', 't']
['s', 'low', 'est']


### Interpretation

The selected character pairs are merged into larger subword units such as `lo`, `low`, `es`, `est`, `ne`, and `new`. This demonstrates how BPE builds a compact subword vocabulary from frequent character sequences.

# TASK2

## Chosen corpus: The Romanian Constitution
https://www.presidency.ro/ro/constitutia-romaniei

In [11]:
def load_corpus(path: str) -> str:
    with open(path, encoding="utf-8") as file:
        return file.read()

### Preprocessing choices

- Convert words to lowercase.
- Split text using whitespace characters.
- Keep punctuation as separate tokens: `.?!,;:-"()`.
- Add `<s>` and `</s>` sentence-boundary markers.
- Detect sentence boundaries after `.`, `?`, `!`, and newline characters.
- Preserve numbers as tokens.
- Do not remove stop words.
- Do not apply stemming or lemmatization.
- Apply the same tokenization policy to the corpus and input sentences.

In [12]:
def tokenize_corpus(corpus: str, insert_sentence_delimeters: bool = True) -> list[str]:
    tokens = []

    end_of_sentence_punctuation = ".?!"
    punctuation = end_of_sentence_punctuation + ",;:-\"()"
    whitespace = " \t\n\r\t\f"
    delimeters = whitespace + punctuation
    word = ""

    new_corpus = f"<s> {corpus} </s> " if insert_sentence_delimeters else f"{corpus} "

    for character in new_corpus:
        if character in delimeters:
            if len(word) > 0:
                tokens.append(word.lower())
                word = ""
            if character in punctuation:
                tokens.append(character)

        else:
            word += character

        # Sentence Delimitation
        if insert_sentence_delimeters and len(tokens) > 0:
            if (tokens[-1] != "<s>"  and character == "\n") or (tokens[-1] in end_of_sentence_punctuation and character.isupper()):
                tokens.extend(["</s>", "<s>"])
    
    return tokens

In [13]:
from IPython.display import clear_output

def input_ngram_number() -> int:
    while True:
        print("Input an 'N' for the N-gram:\n * N = 1 for unigram\n * N = 2 for bigram\n * N = 3 for trigram")
        n = int(input("N = "))
        if n in [1, 2, 3]:
            return n
        else:
            clear_output()
            print(f"N = {n} is not a valid 'N' for an N-gram. Please try again")

In [14]:
def compute_counts(tokens: list[str], ngram_number: int, break_context_on_sentence_end: bool = True) -> dict[tuple[str], dict[str, int]]:
    counts = {}

    if ngram_number not in [1, 2, 3]:
        return counts

    if len(tokens) < ngram_number:
        return counts
    
    context_length = ngram_number - 1
    context: tuple[str] = tuple()
    for token in tokens:
        if len(context) == context_length:
            if not context in counts:
                counts[context] = {}

            if token in counts[context]:
                counts[context][token] += 1
            else:
                counts[context][token] = 1

        if break_context_on_sentence_end and token == "</s>":
            context = tuple()
        elif len(context) < context_length:
            context = context + tuple([token])
        else:
            context = (context[1:] + tuple([token]))[:context_length]

    return counts
    

In [15]:
def compute_probabilities(counts: dict[tuple[str], dict[str, int]], vocab_length: int, apply_laplace_add_one: bool) -> dict[tuple[str], dict[str, float]]:
    probabilities = {}
    for context, token_counts in counts.items():
        probabilities[context] = {}
        context_count = sum(ngram_count for ngram_count in token_counts.values())
        for token, ngram_count in token_counts.items():
            if apply_laplace_add_one:
                probabilities[context][token] = (ngram_count + 1) / (context_count + vocab_length)
            else:
                probabilities[context][token] = ngram_count / context_count

    return probabilities

In [16]:
def get_ngram_probability(probabilities: dict[tuple[str], dict[str, float]], context: tuple[str], token: str, vocab_length: int, apply_laplace_add_one: bool) -> float:
    if context not in probabilities or token not in probabilities[context]:
        if apply_laplace_add_one:
            return 1 / vocab_length
        else:
            return 0
    else:
        return probabilities[context][token]

In [17]:
def count_distinct_ngrams(counts: dict[tuple[str], dict[str, int]]) -> int:
    return sum(len(ngrams) for ngrams in counts.values())

In [18]:
def most_frequent_ngrams(counts: dict[tuple[str], dict[str, int]], top_k: int) -> list[tuple[tuple[str], int]]:
    return sorted(
        (
            (context + tuple([token]), ngram_count) 
                for context, token_counts in counts.items() 
                    for token, ngram_count in token_counts.items()
        ),
        key=lambda x: x[1],
        reverse=True
    )[: top_k]

In [19]:
def compare_probabilities(counts: dict[tuple[str], dict[str, int]], vocab_length: int, ngram_number: int, test_ngram: tuple[str]):
        probabilities = compute_probabilities(counts, vocab_length, apply_laplace_add_one=False)
        probability = get_ngram_probability(probabilities, test_ngram[:-1], test_ngram[-1], vocab_length, apply_laplace_add_one=False)
        print(f"Conditional probability of {ngram_number}-gram {test_ngram} before smoothing: {probability}")
    
        smoothed_probabilities = compute_probabilities(counts, vocab_length, apply_laplace_add_one=True)
        smoothed_probability = get_ngram_probability(smoothed_probabilities, test_ngram[:-1], test_ngram[-1], vocab_length, apply_laplace_add_one=True)
        print(f"Conditional probability of {ngram_number}-gram {test_ngram} after smoothing: {smoothed_probability}")

In [20]:
from pprint import pprint

def main():
    ngram_number = input_ngram_number()

    print(f"\nN = {ngram_number} \n")

    corpus = load_corpus("romanian_constitution.txt")
    corpus_tokens = tokenize_corpus(corpus)
    vocab = set(corpus_tokens)

    print(f"Number of tokens: {len(corpus_tokens)}")
    print(f"Vocabulary size: {len(vocab)}")

    counts = compute_counts(corpus_tokens, ngram_number)

    distinct_ngram_count = count_distinct_ngrams(counts)
    print(f"Number of distinct {ngram_number}-grams: {distinct_ngram_count}")

    topk_frequent_ngrams = most_frequent_ngrams(counts, 20)
    print(f"\nMost frequent {ngram_number}-grams:")
    pprint(topk_frequent_ngrams)

    print()
    compare_probabilities(counts, len(vocab), ngram_number, topk_frequent_ngrams[0][0])
    
    print()
    compare_probabilities(counts, len(vocab), ngram_number, ('elefantul', 'a', 'mers')[:ngram_number])
    
    

In [22]:
main()

Input an 'N' for the N-gram:
 * N = 1 for unigram
 * N = 2 for bigram
 * N = 3 for trigram

N = 2 

Number of tokens: 16984
Vocabulary size: 2578
Number of distinct 2-grams: 8092

Most frequent 2-grams:
[(('.', '</s>'), 537),
 (('<s>', '('), 411),
 (('<s>', 'articolul'), 156),
 (('(', '2'), 138),
 (('2', ')'), 138),
 (('(', '1'), 133),
 (('1', ')'), 133),
 ((',', 'în'), 94),
 (('(', '3'), 89),
 (('3', ')'), 89),
 ((';', '</s>'), 55),
 (('în', 'condiţiile'), 52),
 (('(', '4'), 45),
 (('4', ')'), 45),
 (('şi', 'a'), 41),
 ((',', 'a'), 38),
 (('prin', 'lege'), 37),
 (('de', 'la'), 36),
 ((',', 'cu'), 36),
 (('condiţiile', 'legii'), 36)]

Conditional probability of 2-gram ('.', '</s>') before smoothing: 0.9763636363636363
Conditional probability of 2-gram ('.', '</s>') after smoothing: 0.17199488491048592

Conditional probability of 2-gram ('elefantul', 'a') before smoothing: 0
Conditional probability of 2-gram ('elefantul', 'a') after smoothing: 0.0003878975950349108


### Interpretation

Laplace smoothing assigns a non-zero probability to unseen N-grams, such as `('elefantul', 'a')`, preventing an entire sentence probability from becoming zero.

# TASK3

In [23]:
def input_sentence() -> str:
    max_sentence_length = 200
    while True:
        print("Input a sentence to compute its probability using task2 n-gram model: \n")
        sentence = input("Your sentence= ")

        if not isinstance(sentence, str):
            print("The sentence must be a string. Please try again.")
            continue

        sentence = sentence.strip()

        if len(sentence) == 0:
            print("The sentence cannot be empty. Please try again.")
        elif len(sentence) > max_sentence_length:
            print(f"The sentence cannot be longer than {max_sentence_length} characters. Please try again.")
        else:
            return sentence

In [24]:
def compute_sentence_probability(
    sentence_tokens: list[str],
    probabilities: dict[tuple[str], dict[str, float]],
    ngram_number: int,
    vocab_length: int,
    apply_laplace_add_one: bool
) -> float:
    import math

    sentence_probability = 1.0
    log_probability = 0.0
    context_length = ngram_number - 1
    context: tuple[str] = tuple()

    for token in sentence_tokens:
        if len(context) == context_length:
            token_probability = get_ngram_probability(
                probabilities,
                context,
                token,
                vocab_length,
                apply_laplace_add_one
            )
            sentence_probability *= token_probability
            if token_probability > 0:
                log_probability += math.log(token_probability)
            else:
                log_probability = float("-inf")
            print(f"P({token!r} | {context}) = {token_probability}")

        if token == "</s>":
            context = tuple()
        elif len(context) < context_length:
            context = context + (token,)
        else:
            context = (context[1:] + (token,))[:context_length]

    print(f"Sentence probability: {sentence_probability}")
    print(f"Sentence log-probability: {log_probability}")
    return sentence_probability

In [32]:
ngram_number = input_ngram_number()

corpus = load_corpus("romanian_constitution.txt")
corpus_tokens = tokenize_corpus(corpus)
vocab = set(corpus_tokens)
counts = compute_counts(corpus_tokens, ngram_number)
probabilities = compute_probabilities(
    counts,
    len(vocab),
    apply_laplace_add_one=True
)

sentence = input_sentence()
sentence_tokens = tokenize_corpus(sentence)

print(f"\nSelected N-gram: {ngram_number}")
print(f"Input sentence: {sentence}")
print(f"Tokenized sentence: {sentence_tokens}")
print("\nIndividual conditional probabilities:")

compute_sentence_probability(
    sentence_tokens,
    probabilities,
    ngram_number,
    len(vocab),
    apply_laplace_add_one=True
)

Input an 'N' for the N-gram:
 * N = 1 for unigram
 * N = 2 for bigram
 * N = 3 for trigram
Input a sentence to compute its probability using task2 n-gram model: 


Selected N-gram: 2
Input sentence: românia este un stat național
Tokenized sentence: ['<s>', 'românia', 'este', 'un', 'stat', 'național', '</s>']

Individual conditional probabilities:
P('românia' | ('<s>',)) = 0.000563221627710504
P('este' | ('românia',)) = 0.00423728813559322
P('un' | ('este',)) = 0.0003878975950349108
P('stat' | ('un',)) = 0.0003878975950349108
P('național' | ('stat',)) = 0.0003878975950349108
P('</s>' | ('național',)) = 0.0003878975950349108
Sentence probability: 5.4030087183618626e-20
Sentence log-probability: -44.36474589140528


5.4030087183618626e-20

In [33]:
sentence = input_sentence()
sentence_tokens = tokenize_corpus(sentence)

print(f"\nSelected N-gram: {ngram_number}")
print(f"Input sentence: {sentence}")
print(f"Tokenized sentence: {sentence_tokens}")
print("\nIndividual conditional probabilities:")

compute_sentence_probability(
    sentence_tokens,
    probabilities,
    ngram_number,
    len(vocab),
    apply_laplace_add_one=True
)

Input a sentence to compute its probability using task2 n-gram model: 


Selected N-gram: 2
Input sentence: astăzi vreau să cumpăr lapte
Tokenized sentence: ['<s>', 'astăzi', 'vreau', 'să', 'cumpăr', 'lapte', '</s>']

Individual conditional probabilities:
P('astăzi' | ('<s>',)) = 0.0003878975950349108
P('vreau' | ('astăzi',)) = 0.0003878975950349108
P('să' | ('vreau',)) = 0.0003878975950349108
P('cumpăr' | ('să',)) = 0.0003878975950349108
P('lapte' | ('cumpăr',)) = 0.0003878975950349108
P('</s>' | ('lapte',)) = 0.0003878975950349108
Sentence probability: 3.406453946640685e-21
Sentence log-probability: -47.12861510099479


3.406453946640685e-21

### Interpretation

The sentence probability measures how compatible the input sentence is with the Task 2 corpus. The log-probability provides a numerically stable representation for the very small probability of a longer sentence.

5.4030087183618626e-20 / 3.406453946640685e-21 ≈ 15.86

Therefore, `românia este un stat național` has a probability approximately 15.86 times higher than `astăzi vreau să cumpăr lapte`, according to the 2-gram model.

# TASK4

In [26]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def main():
    model_name = "distilgpt2"
    print(f"Loading tokenizer and model: {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    input_text = "The quick brown fox"
    print(f"\nInput Sequence: '{input_text}'")

    inputs = tokenizer(input_text, return_tensors="pt")
    input_len = inputs["input_ids"].shape[1]
    print(f"Tokenized input IDs shape: {inputs['input_ids'].shape}")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_token_ids = outputs[0][input_len:]
    continuation_text = tokenizer.decode(
        generated_token_ids, skip_special_tokens=True
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\n--- Generation Results ---")
    print(f"Generated Continuation (Next 2 tokens): '{continuation_text}'")
    print(f"Full Generated Text: '{full_text}'")



In [27]:
main()

Loading tokenizer and model: distilgpt2...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


Input Sequence: 'The quick brown fox'
Tokenized input IDs shape: torch.Size([1, 4])

--- Generation Results ---
Generated Continuation (Next 2 tokens): 'es are'
Full Generated Text: 'The quick brown foxes are'


### Interpretation

**DistilGPT-2** causal language model with its associated **GPT-2** tokenizer. Given the input sequence **“The quick brown fox”**, the model generated **“es are”**, producing the complete text **“The quick brown foxes are”**. Two new tokens were generated deterministically using greedy decoding (`do_sample=False`).

In [36]:
test_sentence = "Vulpea maro"
test_sentence_tokens = tokenize_corpus(test_sentence)

print("\n--- N-gram Test ---")
print(f"Input sentence: {test_sentence}")
print(f"Tokenized sentence: {test_sentence_tokens}")
print("\nIndividual conditional probabilities:")

test_sentence_probability = compute_sentence_probability(
    test_sentence_tokens,
    probabilities,
    ngram_number,
    len(vocab),
    apply_laplace_add_one=True
)

print(f"\nFinal sentence probability: {test_sentence_probability}")


--- N-gram Test ---
Input sentence: Vulpea maro
Tokenized sentence: ['<s>', 'vulpea', 'maro', '</s>']

Individual conditional probabilities:
P('vulpea' | ('<s>',)) = 0.0003878975950349108
P('maro' | ('vulpea',)) = 0.0003878975950349108
P('</s>' | ('maro',)) = 0.0003878975950349108
Sentence probability: 5.836483484634121e-11
Sentence log-probability: -23.564307550497396

Final sentence probability: 5.836483484634121e-11


### Comparison with the N-gram model

The Romanian translation of **“The quick brown fox”** is **“Vulpea maro rapidă”**. DistilGPT-2 generated **“es are”** for the English input, while the Romanian N-gram model was tested with the translated sentence and an in-corpus context; the models can produce different continuations because DistilGPT-2 uses neural contextual representations, whereas the N-gram model uses local Romanian corpus counts.

In [37]:
def predict_next_token(
    probabilities: dict[tuple[str], dict[str, float]],
    context: tuple[str],
    vocab_length: int,
    apply_laplace_add_one: bool
) -> tuple[str, float]:
    token_probabilities = probabilities.get(context, {})
    if not token_probabilities:
        raise ValueError(f"The context {context} was not found in the N-gram model.")

    return max(token_probabilities.items(), key=lambda item: item[1])


In [43]:
translated_input = "Vulpea maro rapidă"
translated_tokens = tokenize_corpus(translated_input)
comparison_ngram_number = 2
comparison_counts = compute_counts(corpus_tokens, comparison_ngram_number)
comparison_probabilities = compute_probabilities(
    comparison_counts,
    len(vocab),
    apply_laplace_add_one=True
)
romanian_context = ("este",)
predicted_token, predicted_probability = predict_next_token(
    comparison_probabilities,
    romanian_context,
    len(vocab),
    apply_laplace_add_one=True
)

print("\n--- Pretrained Model vs Romanian N-gram ---")
print("Pretrained model input: 'The quick brown fox'")
print("Pretrained model continuation: 'es are'")
print(f"Romanian translation: '{translated_input}'")
print(f"Romanian translation tokens: {translated_tokens}")
print(f"N-gram in-corpus context: {romanian_context}")
print(f"N-gram predicted next token: '{predicted_token}'")
print(f"N-gram conditional probability: {predicted_probability}")


--- Pretrained Model vs Romanian N-gram ---
Pretrained model input: 'The quick brown fox'
Pretrained model continuation: 'es are'
Romanian translation: 'Vulpea maro rapidă'
Romanian translation tokens: ['<s>', 'vulpea', 'maro', 'rapidă', '</s>']
N-gram in-corpus context: ('este',)
N-gram predicted next token: 'parte'
N-gram conditional probability: 0.0033482142857142855
